# Trading Agent - GPU Experiment Runner
Run the autonomous trading agent on Google Colab with free GPU.

## Setup
1. Upload your `.env` file or set API keys below
2. Run all cells
3. The agent will iterate autonomously

In [ ]:
# Step 1: Clone the repo
!git clone https://github.com/lolokhadim0-source/auto-agent-trading.git
%cd auto-agent-trading
!git checkout feature/trading-agent-v1

In [ ]:
# Step 2: Install dependencies
!pip install anthropic yfinance ccxt alpaca-py fredapi pandas numpy pyarrow ta python-telegram-bot python-dotenv aiohttp

In [ ]:
# Step 3: Set your API keys (fill these in)
import os
os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-YOUR_KEY_HERE'  # REQUIRED
os.environ['TELEGRAM_BOT_TOKEN'] = ''  # Optional
os.environ['TELEGRAM_CHAT_ID'] = ''    # Optional
os.environ['ALPACA_API_KEY'] = ''      # Optional
os.environ['ALPACA_SECRET_KEY'] = ''   # Optional
os.environ['FRED_API_KEY'] = ''        # Optional
os.environ['FINNHUB_API_KEY'] = ''     # Optional
print('API keys set!')

In [ ]:
# Step 4: Download all market data
import sys
sys.path.insert(0, '.')
from data.downloader import download_all
download_all()

In [ ]:
# Step 5: Run the baseline backtest
from strategy.train import strategy
from strategy.backtest import run_full_evaluation, save_result
import logging
logging.basicConfig(level=logging.INFO)

results = run_full_evaluation(strategy)
print(f"\nBaseline composite score: {results['_overall_composite']}")
save_result(results, 'baseline')

In [ ]:
# Step 6: Run the autonomous agent (adjust iterations as needed)
from agent.agent import run

# Run 50 iterations - this will take a while but runs autonomously
run(max_iterations=50, notify=True)

In [ ]:
# Step 7: Check results
import json
from pathlib import Path

best = json.loads(Path('jobs/best_score.json').read_text())
print(f"Best score: {best['score']}")
print(f"Achieved at iteration: {best['iteration']}")
print(f"Timestamp: {best['timestamp']}")

# Show the winning strategy
print('\n--- Current best strategy ---')
print(Path('strategy/train.py').read_text())

In [ ]:
# Step 8: View experiment history
import pandas as pd

history = [json.loads(line) for line in Path('jobs/experiment_history.jsonl').read_text().strip().split('\n')]
df = pd.DataFrame(history)
print(f"Total experiments: {len(df)}")
print(f"Improvements: {df['improved'].sum()}")
print(f"Success rate: {df['improved'].mean()*100:.1f}%")
print(f"\nScore progression:")
print(df[df['improved']][['iteration', 'score', 'description']].to_string())

In [ ]:
# Step 9: Save results back to Google Drive (optional)
from google.colab import drive
drive.mount('/content/drive')

import shutil
save_dir = '/content/drive/MyDrive/trading_agent_results'
os.makedirs(save_dir, exist_ok=True)
shutil.copy('strategy/train.py', f'{save_dir}/best_strategy.py')
shutil.copy('jobs/best_score.json', f'{save_dir}/best_score.json')
shutil.copy('jobs/experiment_history.jsonl', f'{save_dir}/experiment_history.jsonl')
print(f'Results saved to {save_dir}')